# 08 — ML (eye-tracking)

Ensemble de Gradient Boosting (`src/ensemble_gb.py`, média de 360 GB) sobre
`data/processed/dataset_features_et.csv`, avaliado por 10-fold estratificado repetido 20×.
Figuras em `reports/ml/09_et_*`; modelo em `models/gb_ensemble_bcea.pkl`.

In [ ]:
import sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
from ensemble_gb import EnsembleGB

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             roc_auc_score, roc_curve)

DIR_PROCESSED = RAIZ / 'data' / 'processed'
DIR_FIG       = RAIZ / 'reports' / 'ml'
DIR_MODELS    = RAIZ / 'models'
DIR_FIG.mkdir(parents=True, exist_ok=True); DIR_MODELS.mkdir(parents=True, exist_ok=True)

# Mesma paleta do 04_ML_eeg
BLUE_DARK, BLUE_MED, BLUE_LIGHT = '#2C3E50', '#2E86C1', '#AED6F1'
RED_MED    = '#D62728'
GRID_GRAY  = '#B0B0B0'
SPINE_GRAY = '#DDDDDD'
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.edgecolor': SPINE_GRAY})

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.std(ddof=1) ** 2 + (nb - 1) * b.std(ddof=1) ** 2) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else 0.0

EMO  = {'Feliz': 'Happy', 'Neutro': 'Neutral', 'Raiva': 'Angry'}
AOI  = {'Olhos': 'Eyes', 'Nariz': 'Nose', 'Boca': 'Mouth', 'Fora': 'Outside'}
STAT = {'Media': 'Mean', 'Mediana': 'Median', 'Std': 'SD', 'Var': 'Var', 'Min': 'Min', 'Max': 'Max'}
BASE = {'BCEA': 'BCEA', 'Distancia_Total': 'Scanpath Length',
        'Duracao_Media_Visita_Frames': 'Mean Central-AOI Visit (frames)', 'Perc_AOI_Geral': '% Gaze in Central AOI',
        'PrimFix_Latencia_ms': 'First Fixation Latency (ms)', 'PrimFix_Latencia_Olhos_ms': 'First Fixation Latency to Eyes (ms)',
        'Scan_Duracao_Media_Fix_ms': 'Mean Fixation Duration (ms)', 'Scan_Entropia_AOI': 'AOI Entropy',
        'Scan_Perc_Fix_Olhos': '% Fixations on Eyes', 'Scan_Revisitas_Olhos': 'Eye Revisits',
        'Trans_N_Total': 'Total AOI Transitions'}

def prettify(name):
    m = re.match(r'(.*?)_(?:Delta_(Feliz|Neutro)(Neutro|Raiva)|(Feliz|Neutro|Raiva))$', name)
    if not m:
        return name
    base = m.group(1)
    cond = (f' Δ ({EMO[m.group(2)]}−{EMO[m.group(3)]})' if m.group(2) else f' ({EMO[m.group(4)]})')
    if base in BASE:
        txt = BASE[base]
    elif (g := re.match(r'(GazeX|GazeY|Pupila|Velocidade)_(\w+)$', base)):
        txt = {'GazeX': 'Gaze X', 'GazeY': 'Gaze Y', 'Pupila': 'Pupil', 'Velocidade': 'Velocity'}[g[1]] + ' ' + STAT[g[2]]
    elif (g := re.match(r'PrimFix_(\w+)_Perc$', base)):
        txt = f'% First Fixation on {AOI[g[1]]}'
    elif (g := re.match(r'Scan_N_Fix_(\w+)$', base)):
        txt = f'N Fixations on {AOI[g[1]]}'
    elif (g := re.match(r'Trans_(\w+)_(\w+)$', base)):
        txt = f'Transition {AOI[g[1]]}→{AOI[g[2]]}'
    else:
        txt = base.replace('_', ' ')
    return txt + cond


## 1. Modelo

In [ ]:
# ── 1. Dados ────────────────────────────────────────────────────
META = ('ID_Sujeito', 'Grupo', 'Target')
df_et = pd.read_csv(DIR_PROCESSED / 'dataset_features_et.csv')
FEATS_FINAL = [c for c in df_et.columns if c not in META]
X_final  = df_et[FEATS_FINAL].fillna(0).values
y        = df_et['Target'].values
n        = len(y)
sujeitos = df_et['ID_Sujeito'].values
print(f'N = {n}  |  TEA = {(y==1).sum()}  |  Controle = {(y==0).sum()}  |  features = {len(FEATS_FINAL)}')

# ── 2. Validação: 10-fold estratificado repetido 20× (~5 min) ───
N_FOLDS, N_REP = 10, 20

def _fit_pred(tr, te):
    return EnsembleGB().fit(X_final[tr], y[tr]).predict_proba(X_final[te])[:, 1]

tarefas = [(r, tr, te) for r in range(N_REP)
           for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=r).split(X_final, y)]
res = joblib.Parallel(n_jobs=-1)(joblib.delayed(_fit_pred)(tr, te) for _, tr, te in tarefas)
P_final = np.zeros((N_REP, n))                 # P(TEA) fora-da-amostra, uma linha por repetição
for (r, _, te), p in zip(tarefas, res):
    P_final[r, te] = p

# ── 3. Métricas por repetição ───────────────────────────────────
rows = []
for r, p in enumerate(P_final):
    pr = (p > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pr).ravel()
    rows.append(dict(seed=r, accuracy=accuracy_score(y, pr), balanced_accuracy=balanced_accuracy_score(y, pr),
                     sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), roc_auc=roc_auc_score(y, p)))
df_metrics = pd.DataFrame(rows)
df_preds = pd.DataFrame([dict(seed=r, subject=sujeitos[i], y_true=int(y[i]), y_pred=int(P_final[r, i] > 0.5),
                              proba_asd=float(P_final[r, i])) for r in range(N_REP) for i in range(n)])
df_metrics.to_csv(DIR_FIG / '09_et_cv10_metrics_per_rep.csv', index=False)
df_preds.to_csv(DIR_FIG / '09_et_cv10_predictions.csv', index=False)

print('=' * 50)
print(f'  MÉTRICAS DE DESEMPENHO ({N_FOLDS}-FOLD × {N_REP}, média ± DP)')
print('=' * 50)
for c, lab in [('accuracy', 'Acurácia'), ('balanced_accuracy', 'Acurácia bal.'), ('sensitivity', 'Sensibilidade'),
               ('specificity', 'Especificidade'), ('roc_auc', 'ROC-AUC')]:
    print(f'{lab + ":":<16}{df_metrics[c].mean():.4f} ± {df_metrics[c].std():.4f}')
print('=' * 50)

# ── 4. Modelo final: refit com todos os sujeitos + export ───────
ens_final = EnsembleGB().fit(X_final, y)
cols_ens = [FEATS_FINAL[i] for i in ens_final.colunas_usadas()]   # 20 features, em ordem de F
selected = cols_ens[:10]                                          # as 10 primeiras (≥ 60% dos membros)

joblib.dump(ens_final, DIR_MODELS / 'gb_ensemble_bcea.pkl')
meta = {
    'n_sujeitos': int(n), 'n_tea': int((y == 1).sum()), 'n_controle': int((y == 0).sum()),
    'features_entrada': FEATS_FINAL,
    'features_ensemble': cols_ens,
    'grade': {'k_features': list(ens_final.ks), 'max_depth': list(ens_final.depths),
              'learning_rate': list(ens_final.lrs), 'subsample': list(ens_final.subsamples),
              'n_estimators': list(ens_final.n_estimators)},
    'validacao': f'{N_FOLDS}-fold estratificado repetido {N_REP}x',
    **{f'{c}_{s}': float(getattr(df_metrics[c], s)())
       for c in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc'] for s in ['mean', 'std']},
}
(DIR_MODELS / 'gb_ensemble_bcea_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(f'\nModelo salvo em {DIR_MODELS / "gb_ensemble_bcea.pkl"}')

## 2. Plots

### Variabilidade entre as 20 repetições

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
cm_cols = ['roc_auc', 'balanced_accuracy', 'sensitivity', 'specificity']
labels  = ['ROC-AUC', 'Balanced Accuracy', 'Sensitivity', 'Specificity']
data = [df_metrics[c].values for c in cm_cols]
ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.5,
           medianprops=dict(color=BLUE_DARK, linewidth=2),
           boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE_MED),
           whiskerprops=dict(color=BLUE_MED), capprops=dict(color=BLUE_MED))
rng = np.random.default_rng(0)
for i, d in enumerate(data, start=1):
    ax.scatter(np.full(len(d), i) + rng.normal(0, 0.045, len(d)), d,
               color=BLUE_MED, alpha=0.85, s=28, zorder=3)
ax.grid(axis='y', linestyle='--', alpha=0.5, color=GRID_GRAY)
ax.set_ylabel('Score'); ax.set_ylim(0.55, 1.02)
ax.set_title(f'Cross-Validation Performance Variability\n(Stratified {N_FOLDS}-Fold × {N_REP} Repetitions)', fontsize=13)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_cv_boxplot.png', dpi=150, bbox_inches='tight'); plt.show()

### Matriz de confusão (repetição representativa = a de acurácia mais próxima da média)

In [ ]:
seed_repr = int((df_metrics.accuracy - df_metrics.accuracy.mean()).abs().idxmin())
sub = df_preds[df_preds.seed == seed_repr]
cm = confusion_matrix(sub.y_true, sub.y_pred)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold',
                color='white' if cm[i, j] > cm.max() * 0.55 else BLUE_DARK)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Control', 'ASD']); ax.set_yticklabels(['Control', 'ASD'])
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.set_title(f'Aggregated Confusion Matrix\n({N_FOLDS}-Fold, Representative Repetition)', fontsize=13)
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

### Curva ROC média

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
mean_fpr = np.linspace(0, 1, 100)
tprs, aucs = [], []
for sd in df_metrics.seed:
    s = df_preds[df_preds.seed == sd]
    fpr, tpr, _ = roc_curve(s.y_true, s.proba_asd)
    it = np.interp(mean_fpr, fpr, tpr); it[0] = 0.0
    tprs.append(it); aucs.append(roc_auc_score(s.y_true, s.proba_asd))
mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
std_tpr = np.std(tprs, axis=0)
ax.plot(mean_fpr, mean_tpr, color=BLUE_MED, lw=3,
        label=f'Mean ROC (AUC = {np.mean(aucs):.2f} ± {np.std(aucs):.2f})')
ax.fill_between(mean_fpr, np.clip(mean_tpr - std_tpr, 0, 1), np.clip(mean_tpr + std_tpr, 0, 1),
                color=BLUE_LIGHT, alpha=0.3, label='± 1 Standard Deviation')
ax.plot([0, 1], [0, 1], ls='--', lw=2, color='gray', alpha=0.8, label='Chance Level')
ax.grid(linestyle='--', alpha=0.5, color=GRID_GRAY)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'Receiver Operating Characteristic\n(Stratified {N_FOLDS}-Fold × {N_REP} Repetitions)', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_roc_curve.png', dpi=150, bbox_inches='tight'); plt.show()

### Effect size (Cohen's d) das 10 features mais bem ranqueadas

In [ ]:
df_d = (pd.DataFrame([(prettify(f), cohens_d(df_et[f].fillna(0).values[y == 1],
                                             df_et[f].fillna(0).values[y == 0])) for f in selected],
                     columns=['Feature', 'CohensD'])
          .sort_values('CohensD'))
fig, ax = plt.subplots(figsize=(8, 5.5))
norm = TwoSlopeNorm(vmin=min(-1, df_d.CohensD.min() - 0.1), vcenter=0, vmax=max(1, df_d.CohensD.max() + 0.1))
cmap = plt.get_cmap('RdBu_r')
ax.barh(df_d.Feature, df_d.CohensD, color=[cmap(norm(v)) for v in df_d.CohensD], edgecolor='white')
ax.axvline(0, color='#444', lw=1)
ax.set_xlabel("Cohen's d (ASD − Control)")
ax.set_title('Effect Size of the 10 Top-Ranked Features', fontsize=13)
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, pad=0.02).set_label("Cohen's d")
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_effect_sizes.png', dpi=150, bbox_inches='tight'); plt.show()

### Importância média no ensemble e F-score das 20 features

In [ ]:
import matplotlib.patches as mpatches
imp_ens = ens_final.importancias()
f_ens   = ens_final.f_[ens_final.ordem_[:len(cols_ens)]]
df_feat = (pd.DataFrame({'Feature': [prettify(f) for f in cols_ens], 'GB': imp_ens, 'F': f_ens,
                         'BCEA': [f.startswith('BCEA_') for f in cols_ens]})
             .sort_values('GB'))
cor = df_feat['BCEA'].map({True: RED_MED, False: BLUE_MED})
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('GB Ensemble Features (Top 20 by F-score)', fontsize=13, fontweight='bold')
for ax, col, lab in zip(axes, ['GB', 'F'], ['Mean Importance in the Ensemble', 'F-score (ANOVA)']):
    ax.barh(df_feat['Feature'], df_feat[col], color=list(cor), edgecolor='white')
    ax.set_xlabel(lab)
axes[1].set_yticklabels([])
axes[0].legend(handles=[mpatches.Patch(color=RED_MED, label='BCEA'),
                        mpatches.Patch(color=BLUE_MED, label='Other')], fontsize=10, loc='lower right')
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

### SHAP (média entre os 360 membros; 10 features de maior |SHAP|)

In [ ]:
import shap
shap_values, Zs = ens_final.shap_values(X_final)
Xs_df = pd.DataFrame(Zs, columns=[prettify(c) for c in cols_ens])
shap_rank = (pd.Series(np.abs(shap_values).mean(0), index=cols_ens).sort_values(ascending=False))
print(shap_rank.head(10).round(4).to_string())
plt.figure(figsize=(8, 5.5))
shap.summary_plot(shap_values, Xs_df, show=False, plot_size=None, max_display=10)
plt.gcf().suptitle('SHAP — Unimodal Eye-Tracking Model (GB Ensemble)', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_shap_importance.png', dpi=150, bbox_inches='tight'); plt.show()